# Data Ingestion and Initial Cleaning - Online Retail II

## Business Problem
I started this notebook by focusing on a practical business need: I cannot build trustworthy customer segments if the raw transaction data is messy. My goal here was to ingest the retail data, clean the records that would distort customer behavior, and save a reliable transaction table that I could reuse in the rest of the project.

## Why I check requirements first
I like verifying the environment up front so I do not lose time halfway through the notebook because of a missing package. It is a small step, but it makes the workflow more reproducible and easier to hand off.

In [1]:
# Check the core libraries first so the notebook fails early if the environment is incomplete.
required_libraries = ["pandas", "sqlite3", "pathlib"]

print("Requirements check:")
for library_name in required_libraries:
    # sqlite3 and pathlib ship with Python, but I still import them here to confirm the runtime is healthy.
    __import__(library_name)
    print(f"- {library_name}: available")

Requirements check:
- pandas: available
- sqlite3: available
- pathlib: available


## Why I load the dataset dynamically
I wanted this notebook to survive small filename differences without manual edits. By searching the `./data/` folder for likely Online Retail II files, I can keep the workflow flexible while still using a relative path.

In [2]:
# Use explicit imports here to keep the notebook readable and aligned with project standards.
from pathlib import Path
import pandas as pd

# Keep the data path relative so the project is easier to move across machines.
data_folder = Path("./data")

# Search a few common filename patterns because retail datasets often arrive with slightly different naming.
candidate_files = sorted(data_folder.glob("*Online*Retail*II*.csv")) + \
                  sorted(data_folder.glob("*Online*Retail*II*.xlsx")) + \
                  sorted(data_folder.glob("*online_retail_II*.xlsx")) + \
                  sorted(data_folder.glob("*online_retail*.csv"))

if not candidate_files:
    raise FileNotFoundError(
        "No Online Retail II dataset file found in ./data/. "
        "Expected a CSV/XLSX filename containing 'Online' and 'Retail'."
    )

# Load the first matching file and switch readers based on file extension.
dataset_path = candidate_files[0]
if dataset_path.suffix.lower() == ".csv":
    df_retail_raw = pd.read_csv(dataset_path)
else:
    # Default to the first sheet because that is where the Online Retail II transaction table usually lives.
    df_retail_raw = pd.read_excel(dataset_path)

print(f"Loaded dataset: {dataset_path}")
print(f"Raw shape: {df_retail_raw.shape}")

Loaded dataset: data\online_retail_II.xlsx
Raw shape: (525461, 8)


## Why I clean the transactions at this stage
I wanted to remove the obvious records that would damage downstream segmentation before I touched any feature engineering. In this dataset, missing customer identifiers and canceled invoices are not minor issues, because they directly change who looks valuable and who does not.

In [3]:
# Make a copy first so I can always compare my cleaned table back to the raw import if something looks off.
df_retail_cleaned = df_retail_raw.copy()

# Trim the column labels because hidden spaces are an easy way to create avoidable bugs in notebook work.
df_retail_cleaned.columns = df_retail_cleaned.columns.astype(str).str.strip()

# Normalize the column names so the notebook can handle variants like 'Customer ID' and 'CustomerID' without breaking.
normalized_column_map = {
    column_name.lower().replace(" ", "").replace("_", ""): column_name
    for column_name in df_retail_cleaned.columns
}

# Resolve the key fields dynamically because real datasets rarely arrive with perfectly consistent schema naming.
customer_column = normalized_column_map.get("customerid")
invoice_column = normalized_column_map.get("invoice")

if customer_column is None or invoice_column is None:
    raise KeyError(
        "Required columns not found after normalization. "
        f"Available columns: {list(df_retail_cleaned.columns)}"
    )

# Drop rows without a customer identifier because they cannot contribute to customer-level segmentation.
df_retail_cleaned = df_retail_cleaned.dropna(subset=[customer_column])

# Coerce invoices to clean strings before filtering so I do not miss cancellations because of formatting noise.
df_retail_cleaned[invoice_column] = (
    df_retail_cleaned[invoice_column].astype(str).str.strip()
)

# Remove canceled orders here because they would otherwise drag quantity and value signals in the wrong direction.
df_retail_cleaned = df_retail_cleaned[
    ~df_retail_cleaned[invoice_column].str.startswith("C")
]

# Rename these fields to consistent labels so the next notebooks can rely on one clean schema.
df_retail_cleaned = df_retail_cleaned.rename(
    columns={customer_column: "CustomerID", invoice_column: "Invoice"}
)

print(f"Using customer column: {customer_column}")
print(f"Using invoice column: {invoice_column}")
print(f"Cleaned shape: {df_retail_cleaned.shape}")

Using customer column: Customer ID
Using invoice column: Invoice
Cleaned shape: (407695, 8)


## Why I move the cleaned data into SQLite
I did not want every notebook to repeat the ingestion and cleaning logic from scratch. By saving the cleaned transactions into `retail.db`, I gave myself a lightweight, queryable source of truth for the rest of the project.

In [4]:
# Use sqlite3 because it gives me a simple local database without adding infrastructure overhead.
import sqlite3

# Create the database in the project root so the later notebooks can connect to it with a short relative path.
database_path = "retail.db"
sqlite_connection = sqlite3.connect(database_path)

# Overwrite the transactions table on purpose so rerunning the notebook keeps the data source fresh and reproducible.
df_retail_cleaned.to_sql(
    name="transactions",
    con=sqlite_connection,
    if_exists="replace",
    index=False
)

# Close the connection immediately because open database handles are easy to forget in notebook workflows.
sqlite_connection.close()

print(f"Saved cleaned data to {database_path} in table 'transactions'.")

Saved cleaned data to retail.db in table 'transactions'.


## Why I preview the cleaned output
Before I move on, I want one quick sanity check that the cleaning logic actually produced something usable. Looking at a small preview and the data types helps me catch schema issues early instead of discovering them inside the modeling notebook.

In [5]:
# Start with five rows because that is usually enough to spot whether the cleaning logic behaved the way I expected.
display(df_retail_cleaned.head(5))

# Also inspect the dtypes now so I am not surprised later when I start doing date math and spend calculations.
dtype_summary = df_retail_cleaned.dtypes.rename("dtype").to_frame()
display(dtype_summary)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


,dtype
Invoice,str
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[us]
Price,float64
CustomerID,float64
Country,str
